<a href="https://colab.research.google.com/github/Puneet69/Everything-I-Have/blob/main/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq sentence-transformers faiss-cpu pypdf numpy matplotlib scikit-learn -q

In [ ]:
import os
import numpy as np
import faiss

from groq import Groq
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

In [ ]:
from getpass import getpass
GROQ_API_KEY = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [ ]:
client = Groq(api_key=GROQ_API_KEY)

MODEL = "openai/gpt-oss-20b"

In [ ]:
messages = [
    {
        "role":"user",
        "content":"What is inside my private PDF?"
    }
]

response = client.chat.completions.create(
    model = MODEL,
    messages = messages
)

print(response.choices[0].message.content)

I’m sorry, but I can’t access the contents of your private PDF. If you can paste the text or the specific sections you’re interested in, I’ll be happy to help.


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "Embeddings convert text into vectors.",
    "FAISS is used for the similarity search.",
    "Chunking splits large documents into smaller pieces."
]
print(documents)

['RAG stands for Retrieval Augmented Generation.', 'Embeddings convert text into vectors.', 'FAISS is used for the similarity search.', 'Chunking splits large documents into smaller pieces.']


In [ ]:
embeddings = embedding_model.encode(documents)
print(embeddings.shape)
print(embeddings[0])

(4, 384)
[-1.11864574e-01  7.27333128e-02  1.31854666e-02 -1.21591324e-02
 -1.09949321e-01  3.22039984e-02  6.53926209e-02  2.80548763e-02
  9.90313711e-04 -4.80533168e-02  5.87489679e-02  3.41508500e-02
  4.78318371e-02 -7.20908716e-02 -7.90361464e-02  4.43398692e-02
  2.91630831e-02  1.10945180e-01 -4.54712845e-03  2.63821287e-03
 -4.45330180e-02  8.46416131e-02  1.75273027e-02 -3.06797102e-02
  2.60340739e-02  2.63756812e-02 -8.13348293e-02  4.55923844e-03
  1.59398586e-01  4.30702791e-03 -6.03303220e-03  9.03822631e-02
 -4.57265489e-02 -2.21017543e-02 -4.85288203e-02  3.69571447e-02
 -4.90494519e-02  7.59510696e-02  1.24897715e-02  3.49107236e-02
  2.91071534e-02 -3.67275230e-03  2.56013237e-02 -6.01953454e-02
  1.02029284e-02  2.63532754e-02 -2.77443081e-02 -5.00460854e-03
 -5.13045350e-03  1.94188580e-02  3.09097581e-02 -1.39861042e-02
  1.78056024e-02  5.07812612e-02 -2.74632629e-02 -1.56592485e-02
  6.10844083e-02 -8.06052908e-02 -3.48829180e-02  1.13693876e-02
 -5.09755351e-02

In [ ]:
dimension = embeddings.shape[1]
nlist = 2

quantizer = faiss.IndexFlatL2(dimension)

index=faiss.IndexIVFFlat(
    quantizer,
    dimension,
    nlist,
    faiss.METRIC_L2
)
index.train(embeddings)
index.add(embeddings)

In [ ]:
query= "Tell me about Chunking"

query_embeddging = embedding_model.encode([query])

In [ ]:
distances , indices = index.search(
    np.array(query_embeddging).astype("float32"),
    2
)
print(indices)

[[3 1]]


In [ ]:
retrived_docs=[documents[i] for i in indices[0]]
print(retrived_docs)

['Chunking splits large documents into smaller pieces.', 'Embeddings convert text into vectors.']


In [ ]:
context = "\n".join(retrived_docs)
print(context)

Chunking splits large documents into smaller pieces.
Embeddings convert text into vectors.


In [ ]:
messages = [
    {
        "role" : "system",
        "content" : "Answer only using the provided context"
    },
    {
        "role" : "user",
        "content" : f"Context:\n{context}\n\nQuesion:{query}"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages = messages
)

print(response.choices[0].message.content)

Chunking splits large documents into smaller pieces.


In [ ]:
pdf = PdfReader("RAG_Guide_10000_words.pdf")

text = ""
for page in pdf.pages:
    text += page.extract_text()

print(text[:1000])

RAG Test Document - Page 1
 Retrieval-Augmented Generation: A
Comprehensive Guide This document is
 a long-form reference on
Retrieval-Augmented Generation,
usually abbreviated as RAG. It is written
 as test material for RAG pipelines. It
contains many specific, self-contained
 facts, definitions, comparisons, and
 numbered details so that retrieval
quality and answer faithfulness can be
checked with concrete questions. The
guide is organized into twelve chapters,
followed by a glossary and a frequently
 asked questions section.
Chapter 1: Introduction to Retrieval-Augmented
Generation Retrieval-Augmented Generation is a
technique that combines an information retrieval system
with a generative language model. Instead of relying
only on the knowledge stored in the parameters of a
model during training, a RAG system first searches an
external knowledge source for relevant passages and
then supplies those passages to the model as additional
context. The model then writes an answer that is

In [ ]:
chunk_size = 300
chunks= []

for i in range(0, len(text), chunk_size):
  chunk = text[i:i+chunk_size]
  chunks.append(chunk)

print("Total Chunks:",len(chunks))

Total Chunks: 218


In [ ]:
chunk_embeddings = embedding_model.encode(chunks)

In [ ]:
dimension = chunk_embeddings.shape[1]

pdf_index = faiss.IndexFlatL2(dimension)

pdf_index.add(np.array(chunk_embeddings, dtype="float32"))

print(pdf_index.ntotal)

218


In [ ]:
query = "What is the document about?"

query_embedding = embedding_model.encode([query])

In [ ]:
distances , indices = pdf_index.search(
    np.array(query_embeddging, dtype="float32"),
    3
)
retrived_chunks = [chunks[i] for i in indices[0]]

print(retrived_chunks)

[' be unintelligible without the sentences around it.\n3.2 Fixed-size chunking Fixed-size chunking splits text every N characters\nor tokens, often with an overlap. A common starting configuration is 500\nto 1000 tokens per chunk with an overlap of 10 to 20 percent. The overlap\nmakes sure that a sentence', 'unking Chunking\nis the process of splitting documents into pieces that\nare small enough to embed and retrieve precisely yet\nlarge enough to carry meaning. It is one of the most\ninfluential design decisions in a RAG pipeline.\n3.1 Why chunking is needed Embedding models have a maximum input\nlength. Ma', " per chunk, for example five\nsentences with a one sentence overlap. Paragraph-based chunking uses\nthe author's own paragraph breaks as boundaries. Both respect the flow\nof the text and often outperform naive fixed-size splitting on prose\ndocuments.\n3.5 Structure-aware chunking Structure-aware chunki"]


In [ ]:
context = "\n".join(retrived_chunks)

messages = [
    {
        "role" : "system",
        "content" : "Answer only using the provided context"
    },
    {
        "role" : "user",
        "content" : f"Context:\n{context}\n\nQuesion: {query}"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages = messages
)

print(response.choices[0].message.content)

The document discusses different techniques and considerations for splitting texts into manageable pieces—called chunking—for use in embedding and retrieval‑augmented generation (RAG) pipelines. It covers why chunking is necessary, fixed‑size versus structure‑aware chunking, the use of overlaps, and paragraph‑based boundaries.
